<a href="https://colab.research.google.com/github/ZeroOneThree013/BTC_AI_Trading_Strategy/blob/main/BTC_AI_Trading_Strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🪙 AI 預測比特幣價格波動與交易策略系統
> **環境**：Google Colab (T4 GPU) | **前端**：Gradio 4.x

## 📋 執行順序
1. **Cell 1**：安裝套件
2. **Cell 2**：匯入套件與全域狀態
3. **Cell 3**：抓取資料
4. **Cell 4**：特徵工程
5. **Cell 5**：訓練模型（LSTM + XGBoost + SHAP）
6. **Cell 6**：回測引擎
7. **Cell 7**：啟動 Gradio Dashboard

In [1]:
# Cell 1：安裝套件
!pip install -q gradio plotly pandas-ta xgboost shap requests

In [2]:
# Cell 2：匯入套件與全域狀態
import requests, time, pickle, warnings
import numpy as np
import pandas as pd
import pandas_ta as ta
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gradio as gr
import shap
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

np.random.seed(42)
tf.random.set_seed(42)

STATE = dict(df_raw=None, df_features=None, model_lstm=None, model_xgb=None,
             scaler=None, feature_cols=None, df_pred=None,
             df_backtest_lstm=None, df_backtest_xgb=None,
             bt_metrics_lstm=None,  bt_metrics_xgb=None,
             shap_values=None, X_shap=None, eval_results=None,
             trade_log_lstm=[], trade_log_xgb=[])
print("✅ 套件載入完成")

✅ 套件載入完成


In [4]:
# Cell 3：資料抓取（改用 yfinance，不受地區限制）
!pip install -q yfinance

import yfinance as yf
from datetime import datetime

def fetch_btc():
    df = yf.download("BTC-USD", start="2022-01-01", progress=False)
    df = df.reset_index()
    df.columns = [c[0].lower() if isinstance(c, tuple) else c.lower() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"]).dt.date
    df = df[["date","open","high","low","close","volume"]].copy()
    for c in ["open","high","low","close","volume"]:
        df[c] = pd.to_numeric(df[c])
    return df.sort_values("date").reset_index(drop=True)


def fetch_fg(limit=2000):
    r  = requests.get(f"https://api.alternative.me/fng/?limit={limit}&format=json", timeout=10)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["data"])[["timestamp","value"]]
    df["date"]       = pd.to_datetime(df["timestamp"].astype(int), unit="s").dt.date
    df["fear_greed"] = pd.to_numeric(df["value"])
    return df[["date","fear_greed"]].sort_values("date").reset_index(drop=True)


def load_data():
    try:
        print("📡 抓取 BTC-USD（Yahoo Finance）...")
        df = fetch_btc()
        print("📡 抓取 Fear & Greed Index...")
        fg = fetch_fg()
        df = pd.merge(df, fg, on="date", how="left")
        df["fear_greed"] = df["fear_greed"].ffill()
        STATE["df_raw"]  = df
        miss = df.isnull().mean().mean()
        print(f"✅ 完成：{len(df)} 筆 | 缺漏率：{miss:.2%} | {df['date'].min()} ~ {df['date'].max()}")
    except Exception as e:
        print(f"❌ 資料載入失敗：{e}")

load_data()

📡 抓取 BTC-USD（Yahoo Finance）...
📡 抓取 Fear & Greed Index...
✅ 完成：1577 筆 | 缺漏率：0.00% | 2022-01-01 ~ 2026-04-27


In [6]:
# Cell 4：特徵工程
SEQ_LEN = 60

def build_features(df):
    d = df.copy()
    d["rsi"]       = ta.rsi(d["close"], length=14)
    macd           = ta.macd(d["close"])
    d["macd"]      = macd["MACD_12_26_9"]
    d["macd_sig"]  = macd["MACDs_12_26_9"]
    d["macd_hist"] = macd["MACDh_12_26_9"]
    bb             = ta.bbands(d["close"], length=20)
    d["bb_upper"]  = bb.iloc[:, 0]
    d["bb_mid"]    = bb.iloc[:, 1]
    d["bb_lower"]  = bb.iloc[:, 2]
    d["ema7"]      = ta.ema(d["close"], length=7)
    d["ema25"]     = ta.ema(d["close"], length=25)
    d["ema99"]     = ta.ema(d["close"], length=99)
    d["vol_roc"]   = d["volume"].pct_change(5)
    d["fg_ma7"]    = d["fear_greed"].rolling(7).mean()
    d["label"]     = (d["close"].shift(-1) > d["close"]).astype(int)
    before = len(d)
    d = d.dropna().reset_index(drop=True)
    print(f"   移除 NaN：{before - len(d)} 筆，剩餘 {len(d)} 筆")
    return d


def split_scale(df, seq_len=SEQ_LEN):
    FEAT = ["open","high","low","close","volume",
            "rsi","macd","macd_sig","macd_hist",
            "bb_upper","bb_mid","bb_lower",
            "ema7","ema25","ema99","vol_roc","fear_greed","fg_ma7"]
    n, t, v = len(df), int(len(df)*.70), int(len(df)*.85)
    sc  = MinMaxScaler()
    X   = df[FEAT].values.astype(float)
    X[:t] = sc.fit_transform(X[:t])  # fit 僅在訓練集
    X[t:] = sc.transform(X[t:])
    y = df["label"].values

    def seq(X, y, s, e):
        Xs, ys = [], []
        for i in range(s, e - seq_len):
            Xs.append(X[i:i+seq_len]); ys.append(y[i+seq_len])
        return np.array(Xs), np.array(ys)

    Xtr,ytr = seq(X,y,0,t)
    Xva,yva = seq(X,y,t,v)
    Xte,yte = seq(X,y,v,n)

    with open("/content/scaler.pkl","wb") as f: pickle.dump(sc,f)
    STATE.update(scaler=sc, feature_cols=FEAT, df_features=df)
    print(f"✅ 特徵完成 | Train:{Xtr.shape} Val:{Xva.shape} Test:{Xte.shape}")
    return Xtr,ytr,Xva,yva,Xte,yte,FEAT


print("🔧 執行特徵工程...")
df_feat = build_features(STATE["df_raw"])
Xtr,ytr,Xva,yva,Xte,yte,FEAT = split_scale(df_feat)

🔧 執行特徵工程...
   移除 NaN：98 筆，剩餘 1479 筆
✅ 特徵完成 | Train:(975, 60, 18) Val:(162, 60, 18) Test:(162, 60, 18)


In [7]:
# Cell 5：模型訓練（LSTM + XGBoost + SHAP）

def train_lstm(Xtr,ytr,Xva,yva):
    print("🧠 訓練 LSTM...")
    m = Sequential([
        LSTM(128,return_sequences=True,input_shape=(Xtr.shape[1],Xtr.shape[2])),
        Dropout(0.2), LSTM(64), Dropout(0.2),
        Dense(32,activation="relu"), Dense(1,activation="sigmoid")])
    m.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
    m.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=100,batch_size=32,verbose=1,
          callbacks=[EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True),
                     ModelCheckpoint("/content/model_lstm.keras",save_best_only=True)])
    STATE["model_lstm"] = m
    print("✅ LSTM 完成")
    return m


def train_xgb(Xtr,ytr,Xva,yva):
    print("🌲 訓練 XGBoost...")
    # 只取最後一個 timestep，避免 XGBoost 把時序 lag 視為獨立特徵
    m = XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,
                      random_state=42,eval_metric="logloss",
                      early_stopping_rounds=20,verbosity=0)
    m.fit(Xtr[:,-1,:],ytr,
          eval_set=[(Xva[:,-1,:],yva)],verbose=False)
    STATE["model_xgb"] = m
    print("✅ XGBoost 完成")
    return m


def calc_shap(m,Xte):
    print("🔍 計算 SHAP...")
    X2 = Xte[:,-1,:][:200]  # (200, n_features)，與 XGBoost 訓練格式一致
    sv = shap.TreeExplainer(m).shap_values(X2)
    STATE.update(shap_values=sv, X_shap=X2)
    print("✅ SHAP 完成")


def evaluate(Xte,yte):
    res = {}
    if STATE["model_lstm"]:
        prob = STATE["model_lstm"].predict(Xte).flatten()
        p    = (prob > .5).astype(int)
        res["LSTM"] = {"accuracy":accuracy_score(yte,p),"f1":f1_score(yte,p),
                       "auc":roc_auc_score(yte,prob),
                       "pred":p,"prob":prob}
    if STATE["model_xgb"]:
        X2   = Xte[:,-1,:]
        prob = STATE["model_xgb"].predict_proba(X2)[:,1]
        p    = (prob > .5).astype(int)
        res["XGBoost"] = {"accuracy":accuracy_score(yte,p),"f1":f1_score(yte,p),
                          "auc":roc_auc_score(yte,prob),
                          "pred":p,"prob":prob}
    for n,r in res.items():
        print(f"  {n} → Acc:{r['accuracy']:.4f} F1:{r['f1']:.4f} AUC:{r['auc']:.4f}")
    return res


# 執行
train_lstm(Xtr,ytr,Xva,yva)
train_xgb(Xtr,ytr,Xva,yva)
calc_shap(STATE["model_xgb"],Xte)
er = evaluate(Xte,yte)

# 儲存預測
sl = STATE["df_features"].iloc[-len(yte):].copy().reset_index(drop=True)
sl["y_true"]    = yte
sl["pred_lstm"] = er["LSTM"]["pred"];    sl["prob_lstm"] = er["LSTM"]["prob"]
sl["pred_xgb"]  = er["XGBoost"]["pred"]; sl["prob_xgb"]  = er["XGBoost"]["prob"]
STATE.update(df_pred=sl, eval_results=er)
print("\n✅ 全部完成，可繼續執行 Cell 6")

🧠 訓練 LSTM...
Epoch 1/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.4800 - loss: 0.6977 - val_accuracy: 0.4691 - val_loss: 0.6985
Epoch 2/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4851 - loss: 0.6972 - val_accuracy: 0.4753 - val_loss: 0.6936
Epoch 3/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5026 - loss: 0.6961 - val_accuracy: 0.5309 - val_loss: 0.6917
Epoch 4/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4964 - loss: 0.6950 - val_accuracy: 0.5309 - val_loss: 0.6925
Epoch 5/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4954 - loss: 0.6955 - val_accuracy: 0.5309 - val_loss: 0.6923
Epoch 6/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5251 - loss: 0.6932 - val_accuracy: 0.5309 - val_loss: 0.6918
Epoch 7/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4954 - loss: 0.6940 - val_accuracy: 0.5309 - val_loss: 0.6921
Epoch 8/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5118 - loss: 0.6935 - val

In [9]:
# Cell 6：回測引擎

def run_backtest(df_pred, model_type="LSTM", capital=10000.0, fee=0.001, thresh=0.6):
    if capital <= 0: raise ValueError("初始資金必須為正數")
    if fee < 0:      raise ValueError("手續費不可為負數")

    pc   = "prob_lstm" if model_type=="LSTM" else "prob_xgb"
    df   = df_pred.copy().reset_index(drop=True)
    cash, btc = capital, 0.0
    eq, sig, log = [], [], []

    for row in df.itertuples():
        p, price = getattr(row, pc), row.close
        if p > thresh and btc == 0:
            btc = (cash*(1-fee))/price; cash=0
            sig.append("Buy");  log.append({"date":row.date,"action":"Buy","price":price})
        elif p < (1-thresh) and btc > 0:
            cash = btc*price*(1-fee); btc=0
            sig.append("Sell"); log.append({"date":row.date,"action":"Sell","price":price})
        else:
            sig.append("Hold")
        eq.append(cash + btc*price)

    df["signal"]     = sig
    df["equity"]     = eq
    df["bah_equity"] = capital * df["close"] / df["close"].iloc[0]

    ret    = pd.Series(eq).pct_change().dropna()
    sharpe = (ret.mean()/ret.std()*(252**.5)) if ret.std()>0 else 0
    mdd    = ((pd.Series(eq)-pd.Series(eq).cummax())/pd.Series(eq).cummax()).min()
    m      = {"total_return":(eq[-1]-capital)/capital,"sharpe":sharpe,
              "mdd":mdd,"trade_count":len(log)}

    key = model_type.lower()
    STATE[f"df_backtest_{key}"] = df
    STATE[f"bt_metrics_{key}"]  = m
    STATE[f"trade_log_{key}"]   = log
    print(f"✅ 回測完成 ({model_type}) | 報酬:{m['total_return']:.2%} Sharpe:{sharpe:.2f} MDD:{mdd:.2%} 交易:{len(log)}筆")
    return df, m


if STATE["df_pred"] is not None:
    # 先印出機率分布，確認模型輸出範圍
    print("LSTM 機率分布：")
    print(STATE["df_pred"]["prob_lstm"].describe())
    print("\nXGBoost 機率分布：")
    print(STATE["df_pred"]["prob_xgb"].describe())

    run_backtest(STATE["df_pred"], thresh=0.52)
else:
    print("⚠️ 請先完成模型訓練（Cell 5）")

LSTM 機率分布：
count    162.000000
mean       0.517729
std        0.002546
min        0.512047
25%        0.516269
50%        0.518316
75%        0.519427
max        0.521737
Name: prob_lstm, dtype: float64

XGBoost 機率分布：
count    162.000000
mean       0.483765
std        0.011047
min        0.473621
25%        0.473621
50%        0.481136
75%        0.490876
max        0.510924
Name: prob_xgb, dtype: float64
✅ 回測完成 (LSTM) | 報酬:-12.86% Sharpe:-0.36 MDD:-35.31% 交易:1筆


In [10]:
# Cell 7：Gradio Dashboard

# ── 圖表函式 ────────────────────────────────────────

def fig_overview():
    df = STATE["df_raw"]
    if df is None:
        return go.Figure().add_annotation(text="資料載入失敗，請重新執行 Cell 3", showarrow=False)
    fig = make_subplots(specs=[[{"secondary_y":True}]])
    fig.add_trace(go.Scatter(x=df["date"].astype(str),y=df["close"],
                             name="BTC 收盤價",line=dict(color="#F7931A")),secondary_y=False)
    fig.add_trace(go.Scatter(x=df["date"].astype(str),y=df["fear_greed"],
                             name="Fear & Greed",line=dict(color="#6C8EBF",dash="dot")),secondary_y=True)
    fig.update_layout(title="BTC 收盤價 vs Fear & Greed Index",template="plotly_dark",height=430)
    fig.update_yaxes(title_text="Price (USDT)",secondary_y=False)
    fig.update_yaxes(title_text="Fear & Greed (0–100)",secondary_y=True)
    return fig


def fig_predict(model_type):
    df = STATE["df_pred"]
    if df is None:
        return go.Figure().add_annotation(text="請先完成模型訓練（Cell 5）",showarrow=False)
    pc  = "pred_lstm" if model_type=="LSTM" else "pred_xgb"
    x   = df["date"].astype(str)
    fig = make_subplots(rows=2,cols=1,
                        subplot_titles=("預測 vs 實際漲跌","K 線圖與交易訊號"),
                        row_heights=[0.35,0.65])
    fig.add_trace(go.Scatter(x=x,y=df["y_true"],name="實際",line=dict(color="#00D4AA")),row=1,col=1)
    fig.add_trace(go.Scatter(x=x,y=df[pc],name="預測",line=dict(color="#FF6B6B",dash="dash")),row=1,col=1)
    fig.add_trace(go.Candlestick(x=x,open=df["open"],high=df["high"],
                                 low=df["low"],close=df["close"],name="K線"),row=2,col=1)
    dbt = STATE.get(f"df_backtest_{model_type.lower()}")
    if dbt is not None:
        b = dbt[dbt["signal"]=="Buy"];  s = dbt[dbt["signal"]=="Sell"]
        fig.add_trace(go.Scatter(x=b["date"].astype(str),y=b["close"],mode="markers",
                                 name="Buy",marker=dict(color="#00D4AA",symbol="triangle-up",size=10)),row=2,col=1)
        fig.add_trace(go.Scatter(x=s["date"].astype(str),y=s["close"],mode="markers",
                                 name="Sell",marker=dict(color="#FF6B6B",symbol="triangle-down",size=10)),row=2,col=1)
    fig.update_layout(template="plotly_dark",height=580)
    return fig


def fig_backtest(model_type):
    df = STATE.get(f"df_backtest_{model_type.lower()}")
    if df is None:
        return go.Figure().add_annotation(text=f"請先執行 {model_type} 回測",showarrow=False)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["date"].astype(str),y=df["equity"],
                             name="AI 策略",line=dict(color="#F7931A")))
    fig.add_trace(go.Scatter(x=df["date"].astype(str),y=df["bah_equity"],
                             name="Buy-and-Hold",line=dict(color="#6C8EBF",dash="dot")))
    fig.update_layout(title=f"資金曲線：{model_type} AI 策略 vs Buy-and-Hold",
                      template="plotly_dark",height=400,yaxis_title="資產 (USDT)")
    return fig


def fig_shap():
    if STATE["shap_values"] is None:
        return go.Figure().add_annotation(text="請先完成模型訓練（Cell 5）",showarrow=False)
    sv  = STATE["shap_values"]
    fc  = STATE["feature_cols"]
    imp = np.abs(sv).mean(axis=0)  # (n_features,)，XGBoost 已只取最後一個 timestep
    idx = np.argsort(imp)[-10:]
    fig = go.Figure(go.Bar(x=imp[idx],y=[fc[i] for i in idx],
                           orientation="h",marker_color="#F7931A"))
    fig.update_layout(title="SHAP 特徵重要性（前 10）",template="plotly_dark",
                      height=400,xaxis_title="平均 |SHAP|")
    return fig


# ── Gradio 互動函式 ─────────────────────────────────

def act_overview():
    df = STATE["df_raw"]
    if df is None:
        return fig_overview(), "❌ 資料載入失敗，請重新執行 Cell 3"
    miss = df.isnull().mean().mean()
    return fig_overview(), (f"📊 資料筆數：{len(df)} 筆\n"
                            f"📅 時間範圍：{df['date'].min()} ~ {df['date'].max()}\n"
                            f"⚠️ 缺漏率：{miss:.2%}")


def act_predict(model_type):
    if STATE["df_pred"] is None:
        return fig_predict(model_type), "❌ 模型尚未訓練，請先執行 Cell 5"
    er = STATE.get("eval_results",{})
    if model_type not in er:
        return fig_predict(model_type), f"❌ {model_type} 尚未訓練"
    r = er[model_type]
    return fig_predict(model_type), (f"🎯 Accuracy：{r['accuracy']:.4f}\n"
                                     f"📈 F1 Score：{r['f1']:.4f}\n"
                                     f"📊 AUC-ROC：{r['auc']:.4f}")


def act_backtest(model_type, capital, fee, thresh):
    try:
        capital=float(capital); fee_r=float(fee)/100; thresh=float(thresh)
    except Exception:
        return go.Figure(), "❌ 請輸入有效正數"
    if capital<=0 or fee_r<0 or not (0<thresh<1):
        return go.Figure(), "❌ 請輸入有效正數（初始資金 > 0，手續費 ≥ 0，閾值 0–1）"
    if STATE["df_pred"] is None:
        return go.Figure(), "❌ 請先完成模型訓練（Cell 5）"
    try:
        _,m = run_backtest(STATE["df_pred"],model_type,capital,fee_r,thresh)
        note = "" if m["trade_count"]>0 else "\n⚠️ 無交易，建議調整閾值"
        return fig_backtest(model_type), (f"💰 總報酬率：{m['total_return']:.2%}\n"
                                f"📊 Sharpe Ratio：{m['sharpe']:.2f}\n"
                                f"📉 最大回撤：{m['mdd']:.2%}\n"
                                f"🔄 交易次數：{m['trade_count']}{note}")
    except Exception as e:
        return go.Figure(), f"❌ 回測失敗：{e}"


def act_shap():
    if STATE["shap_values"] is None:
        return go.Figure(), "❌ 請先完成模型訓練（Cell 5）"
    return fig_shap(), "✅ SHAP 特徵重要性（基於 XGBoost 最後一個時間步的 18 個特徵）"


# ── Gradio UI ───────────────────────────────────────

with gr.Blocks(theme=gr.themes.Base(), title="BTC AI 交易策略") as demo:
    gr.Markdown("# 🪙 AI 預測比特幣價格波動與交易策略系統")
    gr.Markdown("> 請依序完成 Cell 1–6 後，在此 Dashboard 進行互動分析")

    with gr.Tab("📊 資料總覽"):
        btn1 = gr.Button("🔄 載入資料總覽", variant="primary")
        p1   = gr.Plot()
        t1   = gr.Textbox(label="資料摘要", lines=3)
        btn1.click(act_overview, outputs=[p1, t1])

    with gr.Tab("🤖 模型預測"):
        with gr.Row():
            dd2  = gr.Dropdown(["LSTM","XGBoost"], value="LSTM", label="選擇模型")
            btn2 = gr.Button("🔍 產生預測", variant="primary")
        p2 = gr.Plot()
        t2 = gr.Textbox(label="模型績效", lines=3)
        btn2.click(act_predict, inputs=[dd2], outputs=[p2, t2])

    with gr.Tab("📈 回測報告"):
        with gr.Row():
            dd3  = gr.Dropdown(["LSTM","XGBoost"], value="LSTM", label="選擇模型")
            n3   = gr.Number(value=10000, label="初始資金 (USDT)")
            f3   = gr.Number(value=0.1,   label="手續費率 (%)")
            s3   = gr.Slider(0.5, 0.9, value=0.6, step=0.05, label="訊號閾值")
        btn3 = gr.Button("🚀 執行回測", variant="primary")
        p3   = gr.Plot()
        t3   = gr.Textbox(label="績效指標", lines=5)
        btn3.click(act_backtest, inputs=[dd3,n3,f3,s3], outputs=[p3,t3])

    with gr.Tab("🔍 SHAP 特徵分析"):
        btn4 = gr.Button("📊 產生 SHAP 圖表", variant="primary")
        p4   = gr.Plot()
        t4   = gr.Textbox(label="說明", lines=2)
        btn4.click(act_shap, outputs=[p4, t4])

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ecf32c14f495b252dd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
